# MoR-TurboQuant: Training on WikiText-103

This notebook trains a language model using the MoR-TurboQuant module and compares it against a standard transformer baseline.

**Before running:** Go to Runtime → Change runtime type → Select **T4 GPU** (or A100 if you have Colab Pro)

**What this notebook does:**
1. Installs the mor-tq module from your GitHub repo
2. Downloads WikiText-103 dataset
3. Trains 4 model variants (standard, MoR-only, compression-only, MoR+compression)
4. Compares perplexity, KV memory, and parameter counts
5. Saves results and trained checkpoints

## Step 1: Setup

In [ ]:
# Install dependencies
!pip install -q datasets tiktoken wandb

# Clone your repo (UPDATE THIS URL to your actual repo)
!git clone https://github.com/ipariket/MoR-TurboQuant-Recursion-Aware-Compressed-KV-Cache.git mor-turboquant
!cd mor-turboquant && pip install -e . -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 2: Load WikiText-103 and Build Tokenizer

In [ ]:
from datasets import load_dataset
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# Load WikiText-103
print("Loading WikiText-103...")
raw_dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

# Use tiktoken (GPT-2 tokenizer) — 50257 vocab
enc = tiktoken.get_encoding("gpt2")
VOCAB_SIZE = enc.n_vocab  # 50257
print(f"Vocab size: {VOCAB_SIZE}")

# Tokenize all splits
def tokenize_split(split_name):
    texts = raw_dataset[split_name]["text"]
    all_tokens = []
    for text in texts:
        if text.strip():  # skip empty lines
            all_tokens.extend(enc.encode(text))
    return torch.tensor(all_tokens, dtype=torch.long)

print("Tokenizing train split...")
train_tokens = tokenize_split("train")
print("Tokenizing validation split...")
val_tokens = tokenize_split("validation")
print("Tokenizing test split...")
test_tokens = tokenize_split("test")

print(f"Train: {len(train_tokens):,} tokens")
print(f"Val:   {len(val_tokens):,} tokens")
print(f"Test:  {len(test_tokens):,} tokens")

In [ ]:
# Dataset that chunks tokens into fixed-length sequences
class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        # Drop last incomplete chunk
        self.n_chunks = len(tokens) // seq_len

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        start = idx * self.seq_len
        chunk = self.tokens[start : start + self.seq_len + 1]  # +1 for labels
        x = chunk[:-1]  # input
        y = chunk[1:]   # target (shifted by 1)
        return x, y

SEQ_LEN = 256  # context window
BATCH_SIZE = 32

train_dataset = TokenDataset(train_tokens, SEQ_LEN)
val_dataset = TokenDataset(val_tokens, SEQ_LEN)
test_dataset = TokenDataset(test_tokens, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Seq length: {SEQ_LEN}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Step 3: Define Model Configurations

We train 4 variants to build the comparison table:
1. **Standard Transformer** — 8 unique layers, no routing, FP16 KV
2. **MoR only** — Weight sharing + early exit, but FP16 KV
3. **Standard + 3-bit KV** — 8 unique layers, no routing, but compressed KV
4. **MoR + 3-bit KV (Ours)** — The full module: weight sharing + early exit + compressed KV

In [ ]:
from mor_tq import MoRConfig, MoRModel

# ~30M params each (trainable on T4 in reasonable time)
# For 125M params, increase d_model=768, d_ff=3072, n_recursions=12
# but that needs A100 and ~6 hours

COMMON = dict(
    d_model=512,
    n_heads=8,
    d_ff=2048,
    vocab_size=VOCAB_SIZE,
    max_seq_len=SEQ_LEN,
    dropout=0.1,
)

configs = {
    # 1. Standard: all unique layers, no routing, no compression
    "Standard 8-layer": MoRConfig(
        **COMMON,
        n_recursions=8,
        sharing_strategy="full",
        capacity_factor=1.0,      # keep all tokens (no early exit)
        routing_strategy="expert",
        kv_bits=0,                # no compression
    ),

    # 2. MoR only: weight sharing + routing, but FP16 KV
    "MoR only": MoRConfig(
        **COMMON,
        n_recursions=8,
        sharing_strategy="middle_cycle",
        n_unique_intro=1,
        n_unique_outro=1,
        capacity_factor=0.5,
        routing_strategy="expert",
        kv_bits=0,                # no compression
    ),

    # 3. Standard + compression: all layers, but 3-bit KV
    "Standard + 3-bit KV": MoRConfig(
        **COMMON,
        n_recursions=8,
        sharing_strategy="full",
        capacity_factor=1.0,      # keep all tokens
        routing_strategy="expert",
        kv_bits=3,                # compressed
    ),

    # 4. OURS: MoR + 3-bit KV (the full module)
    "MoR + 3-bit KV (Ours)": MoRConfig(
        **COMMON,
        n_recursions=8,
        sharing_strategy="middle_cycle",
        n_unique_intro=1,
        n_unique_outro=1,
        capacity_factor=0.5,
        routing_strategy="expert",
        kv_bits=3,                # compressed
    ),
}

# Print param counts
print(f"{'Model':<25} {'Params':>12} {'Unique Block':>15}")
print("-" * 55)
for name, cfg in configs.items():
    model = MoRModel(cfg)
    total = sum(p.numel() for p in model.parameters())
    print(f"{name:<25} {total:>12,}")
    del model
    torch.cuda.empty_cache()

## Step 4: Training Loop

In [ ]:
import math
import time
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


@torch.no_grad()
def evaluate(model, loader, device, max_batches=None):
    """Compute perplexity on a dataset."""
    model.eval()
    total_loss = 0
    total_tokens = 0
    kv_stats_sum = defaultdict(float)
    n_batches = 0

    for i, (x, y) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        output = model(x, labels=y)

        total_loss += output.loss.item() * x.shape[0] * x.shape[1]
        total_tokens += x.shape[0] * x.shape[1]

        for k, v in output.kv_stats.items():
            if isinstance(v, (int, float)):
                kv_stats_sum[k] += v
        n_batches += 1

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    avg_kv_stats = {k: v / n_batches for k, v in kv_stats_sum.items()}

    return perplexity, avg_loss, avg_kv_stats


def train_model(
    name,
    config,
    train_loader,
    val_loader,
    device,
    n_epochs=3,
    lr=3e-4,
    router_loss_weight=0.01,
    grad_clip=1.0,
    eval_every=500,
    log_every=100,
):
    """Train a single model variant and return results."""
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    model = MoRModel(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.95),
        weight_decay=0.1,
    )

    # Cosine LR schedule
    total_steps = n_epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, total_steps, eta_min=lr/10)

    # Training
    global_step = 0
    best_val_ppl = float("inf")
    train_losses = []
    val_ppls = []
    start_time = time.time()

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        epoch_tokens = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            output = model(x, labels=y)
            loss = output.loss + router_loss_weight * output.router_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

            epoch_loss += output.loss.item() * x.shape[0] * x.shape[1]
            epoch_tokens += x.shape[0] * x.shape[1]
            global_step += 1

            if global_step % log_every == 0:
                elapsed = time.time() - start_time
                avg_loss = epoch_loss / epoch_tokens
                ppl = math.exp(avg_loss)
                depth = output.exit_depths.float().mean().item()
                active = output.kv_stats.get("active_token_fraction", 1.0)
                lr_now = scheduler.get_last_lr()[0]
                print(f"  Step {global_step:5d} | Loss {avg_loss:.3f} | PPL {ppl:.1f} | "
                      f"Depth {depth:.1f} | Active {active:.0%} | LR {lr_now:.2e} | "
                      f"{elapsed:.0f}s")

            if global_step % eval_every == 0:
                val_ppl, val_loss, val_kv = evaluate(model, val_loader, device, max_batches=50)
                val_ppls.append((global_step, val_ppl))
                print(f"  >>> Val PPL: {val_ppl:.2f} | KV compression: {val_kv.get('compression_vs_standard', 1.0):.1f}x")
                if val_ppl < best_val_ppl:
                    best_val_ppl = val_ppl
                    # Save best checkpoint
                    torch.save(model.state_dict(), f"/content/{name.replace(' ', '_')}_best.pt")
                model.train()

        # End of epoch eval
        val_ppl, val_loss, val_kv = evaluate(model, val_loader, device)
        train_ppl = math.exp(epoch_loss / epoch_tokens)
        print(f"\n  Epoch {epoch+1}/{n_epochs} done | Train PPL: {train_ppl:.2f} | Val PPL: {val_ppl:.2f}")

    # Final test evaluation
    print(f"\nLoading best checkpoint...")
    model.load_state_dict(torch.load(f"/content/{name.replace(' ', '_')}_best.pt"))
    test_ppl, test_loss, test_kv = evaluate(model, test_loader, device)
    param_stats = model.count_parameters()

    total_time = time.time() - start_time

    results = {
        "name": name,
        "test_ppl": test_ppl,
        "best_val_ppl": best_val_ppl,
        "total_params": total_params,
        "param_savings": param_stats["parameter_savings"],
        "kv_compression": test_kv.get("compression_vs_standard", 1.0),
        "active_token_fraction": test_kv.get("active_token_fraction", 1.0),
        "training_time_min": total_time / 60,
        "val_ppls": val_ppls,
    }

    print(f"\n  FINAL: Test PPL={test_ppl:.2f} | KV={test_kv.get('compression_vs_standard', 1.0):.1f}x | "
          f"Params={total_params:,} | Time={total_time/60:.1f}min")

    del model, optimizer
    torch.cuda.empty_cache()

    return results

## Step 5: Train All 4 Variants

This will take ~2-4 hours on T4, or ~1-2 hours on A100.

**If you're short on time**, you can train just 2 variants:
- Comment out the ones you don't need
- Or reduce `n_epochs` to 1

In [ ]:
# Train all variants
# Reduce n_epochs to 1 for a quick test, use 3 for real results
N_EPOCHS = 3

all_results = []

for name, cfg in configs.items():
    result = train_model(
        name=name,
        config=cfg,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        n_epochs=N_EPOCHS,
        lr=3e-4,
        eval_every=500,
        log_every=100,
    )
    all_results.append(result)

print("\n\nAll training complete!")

## Step 6: Results Table

In [ ]:
# Print the comparison table
print("\n" + "=" * 90)
print("RESULTS: WikiText-103 Language Modeling")
print("=" * 90)
print(f"{'Model':<25} {'Test PPL ↓':>10} {'KV Memory':>12} {'Params':>12} {'Param Save':>12} {'Time':>8}")
print("-" * 90)

baseline_kv = None
for r in all_results:
    if baseline_kv is None:
        baseline_kv = 1.0

    kv_str = f"{r['kv_compression']:.1f}x"
    save_str = f"{r['param_savings']:.0%}" if r['param_savings'] > 0.01 else "—"
    time_str = f"{r['training_time_min']:.0f}min"

    print(f"{r['name']:<25} {r['test_ppl']:>10.2f} {kv_str:>12} {r['total_params']:>12,} {save_str:>12} {time_str:>8}")

print("=" * 90)

# Highlight the key finding
if len(all_results) >= 4:
    baseline_ppl = all_results[0]['test_ppl']
    ours_ppl = all_results[3]['test_ppl']
    ours_kv = all_results[3]['kv_compression']
    ppl_diff = ours_ppl - baseline_ppl
    print(f"\nKey finding: MoR + 3-bit KV achieves {ours_kv:.1f}x KV memory reduction")
    print(f"with only {ppl_diff:+.2f} perplexity increase vs standard transformer.")

## Step 7: Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Validation PPL curves
ax1 = axes[0]
for r in all_results:
    steps, ppls = zip(*r['val_ppls']) if r['val_ppls'] else ([], [])
    ax1.plot(steps, ppls, label=r['name'], linewidth=2)
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Validation Perplexity')
ax1.set_title('Training Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Tradeoff — PPL vs KV Memory
ax2 = axes[1]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
for i, r in enumerate(all_results):
    kv_pct = 100 / r['kv_compression'] if r['kv_compression'] > 0 else 100
    ax2.scatter(kv_pct, r['test_ppl'], s=200, c=colors[i], zorder=5, edgecolors='black')
    ax2.annotate(r['name'], (kv_pct, r['test_ppl']),
                 textcoords="offset points", xytext=(10, 10), fontsize=9)

ax2.set_xlabel('KV Memory (% of baseline)')
ax2.set_ylabel('Test Perplexity ↓')
ax2.set_title('Perplexity vs KV Memory Tradeoff')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to /content/results.png")

## Step 8: Save All Results

In [ ]:
import json

# Save results as JSON (for the paper / README)
save_results = []
for r in all_results:
    save_results.append({
        "name": r["name"],
        "test_perplexity": round(r["test_ppl"], 2),
        "best_val_perplexity": round(r["best_val_ppl"], 2),
        "total_parameters": r["total_params"],
        "parameter_savings_pct": round(r["param_savings"] * 100, 1),
        "kv_compression_ratio": round(r["kv_compression"], 1),
        "active_token_fraction_pct": round(r["active_token_fraction"] * 100, 1),
        "training_time_minutes": round(r["training_time_min"], 1),
    })

with open("/content/results.json", "w") as f:
    json.dump(save_results, f, indent=2)

print("Results saved to /content/results.json")
print("\nCopy this into your README:\n")
print(json.dumps(save_results, indent=2))

## Step 9: Download Checkpoints

Run this cell to download the trained model checkpoints and results.

In [ ]:
# Zip everything for download
!zip -j /content/mor_tq_results.zip /content/*_best.pt /content/results.json /content/results.png

from google.colab import files
files.download('/content/mor_tq_results.zip')
print("\nDownload started! Check your Downloads folder.")

---

## What's Next?

After training completes:

1. **Screenshot the results table** from Step 6
2. **Download results.png** — the training curves chart
3. **Update your GitHub README** with the real perplexity numbers
4. **Write the arxiv paper** using these results

If perplexity is within ~1-2 points of baseline with 10x less KV memory, that's publishable.